Creating input data from observations

In [ ]:
import pandas as pd
import numpy as np



observations = pd.DataFrame(pd.read_pickle('/kaggle/input/datasets/maanav0114/harps-n-dataset/observations.pkl'))
pos_stars = list(set(observations[observations['has_exoplanets'] == 1]['star_name']))
neg_stars = list(set(observations[observations['has_exoplanets'] == 0]['star_name']))

# Sinusoidal positional encoding for BJD
NUM_FREQS = 8
min_period = 1.0
max_period = 7300.0
periods = np.logspace(np.log10(min_period), np.log10(max_period), NUM_FREQS)
freqs = 2.0 * np.pi / periods

def bjd_positional_encoding(bjd, ref_bjd):
    dt = bjd - ref_bjd
    encoding = []
    for f in freqs:
        encoding.append(np.sin(f * dt))
        encoding.append(np.cos(f * dt))
    return encoding

pos_inputs = []
neg_inputs = []

for star in pos_stars:
    star_obs = observations[observations['star_name'] == star].sort_values('bjd')
    ref_bjd = star_obs['bjd'].iloc[0]

    rows = []
    for idx in range(len(star_obs)):
        row = list(star_obs.iloc[idx])
        row.pop(0)  # Remove star_name
        row.pop(1)  # Remove rv (keeping rv_centered)
        row.pop(3)  # Remove has_exoplanets label

        bjd = row[0]
        pos_enc = bjd_positional_encoding(bjd, ref_bjd)

        features = [row[3], row[1], row[2]] + pos_enc
        rows.append(features)

    pos_inputs.append(np.array(rows))

for star in neg_stars:
    star_obs = observations[observations['star_name'] == star].sort_values('bjd')
    ref_bjd = star_obs['bjd'].iloc[0]

    rows = []
    for idx in range(len(star_obs)):
        row = list(star_obs.iloc[idx])
        row.pop(0)
        row.pop(1)
        row.pop(3)

        bjd = row[0]
        pos_enc = bjd_positional_encoding(bjd, ref_bjd)

        features = [row[3], row[1], row[2]] + pos_enc
        rows.append(features)

    neg_inputs.append(np.array(rows))

print(f"Positive stars: {len(pos_inputs)}, total observations: {sum(len(x) for x in pos_inputs)}")
print(f"Negative stars: {len(neg_inputs)}, total observations: {sum(len(x) for x in neg_inputs)}")
print(f"Feature dimension per observation: {pos_inputs[0].shape[1]}")

# ── Feature standardization (fit on ALL data, apply uniformly) ──
# Stack every observation into one big matrix, compute per-feature mean/std
all_obs = np.concatenate(pos_inputs + neg_inputs, axis=0)  # (N_total, 19)
feat_mean = all_obs.mean(axis=0)  # (19,)
feat_std = all_obs.std(axis=0)    # (19,)

# Clamp std to avoid division by zero (positional encodings are bounded, rv_err is always >0)
feat_std = np.clip(feat_std, 1e-8, None)

# Apply standardization: (x - mean) / std
pos_inputs = [(star - feat_mean) / feat_std for star in pos_inputs]
neg_inputs = [(star - feat_mean) / feat_std for star in neg_inputs]

print(f"\nAfter standardization:")
print(f"  Feature means: {np.concatenate(pos_inputs + neg_inputs).mean(axis=0)[:3]}")  # first 3
print(f"  Feature stds:  {np.concatenate(pos_inputs + neg_inputs).std(axis=0)[:3]}")

Creating Data Splits

In [ ]:
import torch
from torch.utils.data import DataLoader, Dataset 
from sklearn.model_selection import train_test_split 
import random



def seed_everything(seed):
    import random
    import torch 
    import numpy as np

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True


seed = 42
seed_everything(seed)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")


train_pos, temp_pos = train_test_split(pos_inputs, test_size = 0.4, random_state = seed)
val_pos, test_pos = train_test_split(temp_pos, test_size = 0.5, random_state = seed)

train_neg, temp_neg = train_test_split(neg_inputs, test_size = 0.4, random_state = seed)
val_neg, test_neg = train_test_split(temp_neg, test_size = 0.5, random_state = seed)


train_data = train_pos + train_neg
val_data = val_pos + val_neg
test_data = test_pos + test_neg

In [ ]:
class StarDataset(Dataset):
    def __init__(self, stars, labels):
        self.data = stars
        self.labels = labels

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        star = torch.tensor(self.data[idx], dtype=torch.float32)
        label = torch.tensor(self.labels[idx], dtype=torch.float32)
        return star, label

def collate_stars(batch):
    """Custom collate: pad variable-length star sequences to the longest in the batch.

    Returns:
        padded:  (B, max_seq_len, 19) — feature tensors, zero-padded
        mask:    (B, max_seq_len)     — True where real data is, False where padding is
        labels:  (B,)                 — labels
    """
    stars, labels = zip(*batch)

    # Find the longest sequence in this batch
    max_len = max(s.shape[0] for s in stars)

    padded = []
    mask = []

    for star in stars:
        seq_len = star.shape[0]
        pad_len = max_len - seq_len
        feat_dim = star.shape[1]  # 19

        # Pad with zeros along the time dimension
        if pad_len > 0:
            padding = torch.zeros(pad_len, feat_dim)
            padded_star = torch.cat([star, padding], dim=0)
        else:
            padded_star = star

        # Mask: True = real observation, False = padding
        star_mask = torch.cat([
            torch.ones(seq_len),
            torch.zeros(pad_len)
        ])

        padded.append(padded_star)
        mask.append(star_mask)

    padded = torch.stack(padded)     # (B, max_seq_len, 19)
    mask = torch.stack(mask)         # (B, max_seq_len)
    labels = torch.stack(labels)     # (B,)

    return padded, mask, labels

train_labels = [1 for i in range(len(train_pos))] + [0 for i in range(len(train_neg))]
val_labels = [1 for i in range(len(val_pos))] + [0 for i in range(len(val_neg))]
test_labels = [1 for i in range(len(test_pos))] + [0 for i in range(len(test_neg))]

train_dataset = StarDataset(train_data, train_labels)
val_dataset = StarDataset(val_data, val_labels)
test_dataset = StarDataset(test_data, test_labels)

In [ ]:
batch_size = 32

train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=batch_size,
    shuffle=True,
    drop_last=False,
    collate_fn=collate_stars,    # <-- the key addition
    pin_memory=True if device.type == 'cuda' else False
)

val_loader = DataLoader(
    dataset=val_dataset,
    batch_size=batch_size,
    shuffle=False,
    drop_last=False,
    collate_fn=collate_stars,
    pin_memory=True if device.type == 'cuda' else False
)

test_loader = DataLoader(
    dataset=test_dataset,
    batch_size=batch_size,
    shuffle=False,
    drop_last=False,
    collate_fn=collate_stars,
    pin_memory=True if device.type == 'cuda' else False
)

Creating Model

In [ ]:
import torch.nn as nn



class ExoplanetTransformer(nn.Module):
    def __init__(self, feat_dim=19, d_model=64, nhead=4, num_layers=2, dim_feedforward=128, dropout=0.1):
        super().__init__()

        # Project input features to Transformer hidden dimension
        self.input_proj = nn.Linear(feat_dim, d_model)

        # Transformer encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True  # (B, seq_len, d_model) instead of (seq_len, B, d_model)
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        # Classification head: pooled output -> probability
        self.classifier = nn.Sequential(
            nn.Linear(d_model, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 1),       # raw logits out, no sigmoid
        )

    def forward(self, x, mask):
        """
        Args:
            x:    (B, seq_len, 19) — padded feature sequences
            mask: (B, seq_len)     — 1 = real data, 0 = padding

        Returns:
            (B,) — probability of hosting an exoplanet
        """
        # Project 19-dim input to d_model dimensions
        x = self.input_proj(x)  # (B, seq_len, d_model)

        # Transformer: PyTorch wants True = IGNORE, but our mask is True = REAL
        src_key_padding_mask = ~mask.bool()  # invert: True where padding is
        x = self.transformer(x, src_key_padding_mask=src_key_padding_mask)
        # x: (B, seq_len, d_model)

        # Masked mean pool: average only over real observations, ignore padding
        mask_expanded = mask.unsqueeze(-1).float()  # (B, seq_len, 1)
        x = (x * mask_expanded).sum(dim=1) / mask_expanded.sum(dim=1)  # (B, d_model)

        # Classify
        out = self.classifier(x).squeeze(-1)  # (B,)
        return out

model = ExoplanetTransformer().to(device)
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")

Training model

In [ ]:
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR



# Class-weighted loss for the ~1:5 imbalance
n_pos = len(train_pos)
n_neg = len(train_neg)
pos_weight = torch.tensor([n_neg / n_pos]).to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
# Note: BCEWithLogitsLoss expects raw logits, so we'll remove Sigmoid from the model
# and apply it only at inference time

optimizer = Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = CosineAnnealingLR(optimizer, T_max=50)

train_losses = []
val_losses = []
train_accs = []
val_accs = []

num_epochs = 50

for epoch in range(num_epochs):
    # Train
    model.train()
    train_loss = 0
    train_correct = 0
    train_total = 0

    for padded, mask, labels in train_loader:
        padded = padded.to(device)
        mask = mask.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        logits = model(padded, mask)  # raw logits, no sigmoid
        loss = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0) # Gradient Clipping
        optimizer.step()

        train_loss += loss.item() * padded.size(0)
        preds = (logits > 0).float()
        train_correct += (preds == labels).sum().item()
        train_total += padded.size(0)

    scheduler.step()

    # Validate
    model.eval()
    val_loss = 0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for padded, mask, labels in val_loader:
            padded = padded.to(device)
            mask = mask.to(device)
            labels = labels.to(device)

            logits = model(padded, mask)
            loss = criterion(logits, labels)

            val_loss += loss.item() * padded.size(0)
            preds = (logits > 0).float()
            val_correct += (preds == labels).sum().item()
            val_total += padded.size(0)


    epoch_train_loss = train_loss / train_total
    epoch_val_loss = val_loss / val_total
    train_acc = train_correct / train_total
    val_acc = val_correct / val_total
    train_losses.append(epoch_train_loss)
    val_losses.append(epoch_val_loss)
    train_accs.append(train_acc)
    val_accs.append(val_acc)
    print(f"Epoch {epoch+1:3d}/{num_epochs} | Train Loss: {train_loss/train_total:.4f} Acc: {train_acc:.4f} | Val Loss: {val_loss/val_total:.4f} Acc: {val_acc:.4f} | LR: {scheduler.get_last_lr()[0]:.6f}")

In [ ]:
torch.save(model.state_dict(), "/kaggle/working/model.pth")

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Loss plot
ax1.plot(train_losses, label='Train Loss', color='#1f77b4')
ax1.plot(val_losses, label='Val Loss', color='#ff7f0e')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training vs Validation Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Accuracy plot
ax2.plot(train_accs, label='Train Accuracy', color='#1f77b4')
ax2.plot(val_accs, label='Val Accuracy', color='#ff7f0e')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.set_title('Training vs Validation Accuracy')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

Quick Test (Temp)

In [ ]:
model.eval()
with torch.no_grad():
    for stars, masks, labels in test_loader:
        # Grab just the first star from the batch
        star = stars[0:1].to(device)    # (1, seq_len, 19) — keeps batch dim
        mask = masks[0:1].to(device)    # (1, seq_len)
        label = labels[0:1].to(device)  # (1,)

        logit = model(star, mask)
        prob = torch.sigmoid(logit).item()

        print(f"P(exoplanet) = {prob:.4f}  Actual: {label.item():.0f}")
        break